In [26]:
import pandas as pd
import numpy as np
import google.generativeai as genai
import os
from dotenv import load_dotenv

In [27]:
load_dotenv()

True

In [28]:
api_key = os.getenv("GEMINI_API_KEY")

In [29]:
genai.configure(api_key=api_key)

In [30]:
def load_data(file_path):
    """
    Load CSV file into a Pandas DataFrame.

    Args:
        file_path (str): Path to CSV file

    Returns:
        pd.DataFrame: Loaded dataframe
    """
    try:
        df = pd.read_csv(file_path)
        print("Data loaded successfully.")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None


In [31]:
df = load_data("sample.csv")
df.head()

Data loaded successfully.


,OrderID,Date,Region,Product,Category,Sales,Quantity,Profit,Customer
0,1001,2024-01-05,North,Laptop,Electronics,80000.0,2,12000.0,Amit
1,1002,2024-01-07,South,Shirt,Clothing,1500.0,3,300.0,Riya
2,1003,2024-01-08,East,Mobile,Electronics,20000.0,1,4000.0,Rahul
3,1004,2024-01-10,West,Sofa,Furniture,30000.0,1,5000.0,Neha
4,1005,2024-01-12,North,Table,Furniture,NaN,2,2000.0,Arjun


In [32]:
def clean_data(df):
    df = df.copy()

    # Remove duplicates
    df = df.drop_duplicates()

    # Handle missing values
    for col in df.columns:
        if df[col].dtype in ["float64", "int64"]:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "Unknown")

    # Convert numeric-like columns
    for col in df.columns:
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notna().sum() > 0:
                df[col] = converted
        except:
            pass

    # Convert datetime-like columns
    for col in df.select_dtypes(include=["object", "string"]).columns:
        try:
            converted = pd.to_datetime(df[col], errors='coerce')
            if converted.notna().sum() > 0:
                df[col] = converted
        except:
            pass

    print("Data cleaned successfully.")
    return df

In [33]:
cleaned_df = clean_data(df)
cleaned_df.head()

Data cleaned successfully.


/var/folders/1p/p9dkl6c941g3v1_cf0km0tkm0000gn/T/ipykernel_50905/2040979641.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors='coerce')
/var/folders/1p/p9dkl6c941g3v1_cf0km0tkm0000gn/T/ipykernel_50905/2040979641.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors='coerce')
/var/folders/1p/p9dkl6c941g3v1_cf0km0tkm0000gn/T/ipykernel_50905/2040979641.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors='coerce')
/var/folders/1p/p9dkl6c941g3v1_cf0

,OrderID,Date,Region,Product,Category,Sales,Quantity,Profit,Customer
0,1001,2024-01-05,North,Laptop,Electronics,80000.0,2,12000.0,Amit
1,1002,2024-01-07,South,Shirt,Clothing,1500.0,3,300.0,Riya
2,1003,2024-01-08,East,Mobile,Electronics,20000.0,1,4000.0,Rahul
3,1004,2024-01-10,West,Sofa,Furniture,30000.0,1,5000.0,Neha
4,1005,2024-01-12,North,Table,Furniture,22000.0,2,2000.0,Arjun


In [34]:
def basic_analysis(df):
    """
    Perform basic analysis:
    - Summary statistics
    - Column data types
    - Null value counts

    Args:
        df (pd.DataFrame)

    Returns:
        dict
    """
    analysis = {
        "shape": df.shape,
        "columns": df.columns.tolist(),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "null_counts": df.isnull().sum().to_dict(),
        "summary": df.describe(include="all").to_dict()
    }

    print("Basic analysis completed.")
    return analysis

In [35]:
analysis = basic_analysis(cleaned_df)
analysis


Basic analysis completed.


{'shape': (10, 9),
 'columns': ['OrderID',
  'Date',
  'Region',
  'Product',
  'Category',
  'Sales',
  'Quantity',
  'Profit',
  'Customer'],
 'dtypes': {'OrderID': 'int64',
  'Date': 'datetime64[us]',
  'Region': 'str',
  'Product': 'str',
  'Category': 'str',
  'Sales': 'float64',
  'Quantity': 'int64',
  'Profit': 'float64',
  'Customer': 'str'},
 'null_counts': {'OrderID': 0,
  'Date': 0,
  'Region': 0,
  'Product': 0,
  'Category': 0,
  'Sales': 0,
  'Quantity': 0,
  'Profit': 0,
  'Customer': 0},
 'summary': {'OrderID': {'count': 10.0,
   'unique': nan,
   'top': nan,
   'freq': nan,
   'mean': 1005.5,
   'min': 1001.0,
   '25%': 1003.25,
   '50%': 1005.5,
   '75%': 1007.75,
   'max': 1010.0,
   'std': 3.0276503540974917},
  'Date': {'count': 10,
   'unique': nan,
   'top': nan,
   'freq': nan,
   'mean': Timestamp('2024-01-14 04:48:00'),
   'min': Timestamp('2024-01-05 00:00:00'),
   '25%': Timestamp('2024-01-08 12:00:00'),
   '50%': Timestamp('2024-01-13 12:00:00'),
   '75%':

In [36]:
def generate_insights(df):
    """
    Generate insights using Gemini API based on dataframe summary.

    Args:
        df (pd.DataFrame)

    Returns:
        str: Generated insights
    """
    try:
        summary = basic_analysis(df)

        prompt = f"""
        You are a data analyst. Based on the following dataset summary, generate key insights:

        {summary}

        Focus on:
        - Trends
        - Patterns
        - Anomalies
        - Business insights
        """

        model = genai.GenerativeModel("gemini-2.5-flash")
        response = model.generate_content(prompt)

        print("Insights generated successfully.")
        return response.text

    except Exception as e:
        return f"Error generating insights: {e}"

In [37]:
insights = generate_insights(cleaned_df)
print(insights)

Basic analysis completed.
Insights generated successfully.
Based on the provided dataset summary, here are the key insights:

**Overall Data Quality & Scope:**
*   The dataset is very small, containing only 10 transactions, all of which are complete with no missing values.
*   Data spans a relatively short period, from January 5th to January 25th, 2024.

---

**1. Trends & Patterns:**

*   **Dominant Category & Product:** "Electronics" is the most frequent product category (40% of transactions), with "Laptop" being the most frequently sold individual product (20% of transactions). This suggests a strong focus or demand within the electronics segment.
*   **Regional Activity:** The "North" region accounts for the highest number of transactions (30% of the total) in this sample, potentially indicating a key market.
*   **Low Quantity per Order:** The average `Quantity` sold per order is very low (1.5 items), with a maximum of 3. This indicates that customers typically purchase only one o

In [38]:
# -----------------------------------------
# Cell 6 (Updated): suggest_visualizations(df)
# -----------------------------------------
def suggest_visualizations(df):
    """
    Suggest appropriate visualizations based on column types.

    Args:
        df (pd.DataFrame)

    Returns:
        list: Suggested visualizations
    """
    suggestions = []

    # Numeric columns
    numeric_cols = df.select_dtypes(include=np.number).columns

    # FIX: include both 'object' and 'string'
    categorical_cols = df.select_dtypes(include=["object", "string"]).columns

    # Numeric columns
    if len(numeric_cols) > 0:
        suggestions.append("Histogram for numeric distributions")
        suggestions.append("Boxplot for outlier detection")

    # Categorical columns
    if len(categorical_cols) > 0:
        suggestions.append("Bar chart for category counts")

    # Mixed relationships
    if len(numeric_cols) >= 2:
        suggestions.append("Correlation heatmap")
        suggestions.append("Scatter plot between numeric variables")

    print("Visualization suggestions generated.")
    return suggestions

In [39]:
viz_suggestions = suggest_visualizations(cleaned_df)
viz_suggestions

Visualization suggestions generated.


['Histogram for numeric distributions',
 'Boxplot for outlier detection',
 'Bar chart for category counts',
 'Correlation heatmap',
 'Scatter plot between numeric variables']